# Final capstone: reproducible optimisation evidence

**Imperial College ML & AI — Stage 2, Component 25.3.** This notebook reproduces the main quantitative claims, with comments explaining each decision. It does **not** pretend to replay every historical optimisation decision.

**Inputs:** [279 recorded observations](../data/all_observations.csv), [final bests](../reports/current_best_after_week13.csv), and the project's Python modules.

**Scope:** verify provenance and counts; chart progress over 13 rounds; demonstrate a GP-based candidate suggestion without assigning an unobserved score; recover F8 using **only the original 40 observations** and check **13 later scores**; verify the permitted F5 boundary result.

The original HEBO-inspired and TuRBO-style code is [here](../src/bbo/optimizer.py). Full HEBO and BoTorch TuRBO-1 were added as **post-capstone research**, not used to generate these portal returns.

In [ ]:
# Make the notebook work when Jupyter starts in the repository root OR notebooks/.
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

cwd = Path.cwd().resolve()
ROOT = next((p for p in [cwd, *cwd.parents] if (p / 'data/all_observations.csv').exists()), None)
if ROOT is None:
    raise FileNotFoundError("Clone the repository and open the notebook from inside it.")
sys.path.insert(0, str(ROOT / 'src'))

from bbo.structural_validation import recover_f8, f5_at
from bbo.surrogates import fit_gp, inverse_mean
from bbo.acquisition import expected_improvement
from bbo.search import sobol_points, portal_string

print("Repository root:", ROOT)

## 1. Dataset and missing values

I start with the exact historical observations, not model-generated guesses. The programme supplied 175 starting points, followed by my 104 returned portal queries. One returned F7 score from Round 11 has no retained input vector, so it remains included in performance counts but is not used for fitting or spatial plots.

In [ ]:
observations = pd.read_csv(ROOT / 'data/all_observations.csv')
required = {'function', 'round', 'source', 'input_status', 'output'}
assert required.issubset(observations.columns)
assert len(observations) == 279
assert (observations['source'] == 'course_initial').sum() == 175
assert (observations['source'] == 'personal_portal').sum() == 104
assert observations['output'].notna().all()

missing = observations.loc[observations['input_status'] == 'missing_input']
assert len(missing) == 1
assert int(missing.iloc[0]['function']) == 7 and int(missing.iloc[0]['round']) == 11
assert missing.iloc[0][[f'x{i}' for i in range(1, 7)]].isna().all()

display(pd.DataFrame({
    'Dataset': ['Course initial', 'Personal portal', 'Total', 'Rows with missing input'],
    'Records': [175, 104, len(observations), len(missing)]
}))
display(observations.groupby('function', as_index=False)['output'].agg(['count','max']))

## 2. Verify the final bests

A final result must be the **largest observed return** for that function, not the latest return or a GP prediction. I recompute all eight maxima from the complete record and compare them with the published results table.

In [ ]:
published = pd.read_csv(ROOT / 'reports/current_best_after_week13.csv')
computed = observations.groupby('function')['output'].max().sort_index()
stored = published.set_index('Function')['Final_Best'].sort_index()
np.testing.assert_allclose(computed.to_numpy(), stored.to_numpy(), atol=1e-12, rtol=1e-12)

view = published[['Function', 'Final_Best', 'First_Achieved_Round', 'Best_Input']]
display(view)
print("All eight final bests reproduce exactly within floating-point tolerance.")

## 3. Progress through thirteen rounds

For every function I take the best output of each round and calculate its cumulative maximum. Each subplot has its **own y-axis scale**: scores across different functions cannot be added or compared directly.

The first round in this notebook is `round=0`, representing the programme's starting data. The remaining 13 rounds are my returned queries.

In [ ]:
fig, axes = plt.subplots(4, 2, figsize=(12, 13))
for function_id, axis in zip(range(1, 9), axes.flat):
    records = observations.loc[observations['function'] == function_id]
    best_by_round = records.groupby('round')['output'].max().sort_index().cummax()
    axis.plot(best_by_round.index, best_by_round.to_numpy(), marker='o', linewidth=1.7)
    axis.set(title=f'F{function_id}: best returned score so far', xlabel='Round', ylabel='Output')
    axis.grid(alpha=0.25)
plt.tight_layout()
plt.show()

## 4. Illustrative Gaussian Process decision

This is a **separate retrospective example**, not a reconstruction of the exact candidate chosen in the course. I use F2's observations available **through Round 9**. A Matérn 2.5 GP, fitted with a Yeo–Johnson response transformation, predicts scores and uncertainty across a Sobol candidate pool. Expected Improvement (EI) balances predicted gains and uncertainty.

The algorithm proposes an input only. Without querying the real portal at that exact input, **its true score is unknown**.

In [ ]:
history_f2 = observations.loc[(observations['function'] == 2) & (observations['round'] <= 9)]
X = history_f2[['x1', 'x2']].to_numpy(float)
y = history_f2['output'].to_numpy(float)

# No later observations are available to this illustrative Week 9 decision.
model, transform, transformed_y = fit_gp(
    X, y, kernel='Matern25', transform='yeo-johnson', seed=25, restarts=0
)
pool = sobol_points(d=2, n_power=9, seed=25)
mu, sd = model.predict(pool, return_std=True)
ei = expected_improvement(mu, sd, float(transformed_y.max()))
winner = int(np.argmax(ei))
proposal = pool[winner]
predicted_raw_scale = float(inverse_mean([mu[winner]], transform)[0])

print('Illustrative, unsubmitted candidate:', portal_string(proposal))
print('GP prediction (inverse-transformed location):', round(predicted_raw_scale, 6))
print('Predicted uncertainty on transformed scale:', round(float(sd[winner]), 6))
print('True portal score: UNKNOWN (not measured)')

## 5. F8: test a simple structural model against later evidence

F8's model is an **additive quadratic with 17 coefficients**. The fitting routine uses the **40 course-provided points only**, then predicts the next **13 personal portal outputs** without fitting to them.

This test was conducted retrospectively after the capstone. It demonstrates that a simple equation already fitted the initial evidence exceptionally well; it does not claim I had identified it in Round 1.

In [ ]:
f8 = recover_f8(ROOT)
assert f8['rank'] == 17
assert f8['initial_count'] == 40 and f8['holdout_count'] == 13
assert f8['max_holdout_error'] < 2e-14
assert abs(f8['maximum'] - 10) < 1e-12

print('Full-rank coefficients:', f8['rank'])
print('Fitted optimum coordinates:', np.round(f8['centre'], 6))
print('Fitted optimum score:', round(f8['maximum'], 12))
print('Maximum absolute prediction error on 13 later returns:',
      f"{f8['max_holdout_error']:.3e}")
print('Curvature ratio, x3 to x8:', f8['weights'][2] / f8['weights'][7])

### Visualise the F8 equation

Because F8 has **eight inputs**, this plot shows a **two-input slice**, not the entire function. Only x3 and x8 vary; all six remaining values stay at the fitted optimum. This makes their different curvature weights visible.

In [ ]:
xs = np.linspace(0, .999999, 100)
x3, x8 = np.meshgrid(xs, xs)
# The additive model is f(x) = peak - sum(weight_j * (x_j - centre_j)**2).
# Fixing the other six coordinates at centre makes their squared terms zero.
surface = (f8['maximum']
           - f8['weights'][2] * (x3 - f8['centre'][2])**2
           - f8['weights'][7] * (x8 - f8['centre'][7])**2)

fig, ax = plt.subplots(figsize=(7, 5))
contours = ax.contourf(x3, x8, surface, levels=22)
ax.scatter([f8['centre'][2]], [f8['centre'][7]], marker='x', s=100, label='Fitted vertex')
ax.set(xlabel='x3 (variable)', ylabel='x8 (variable)',
       title='F8 reconstructed quadratic: other six inputs fixed')
ax.legend()
fig.colorbar(contours, ax=ax, label='Fitted score')
plt.tight_layout()
plt.show()

## 6. F5: check the legal boundary

The reconstructed F5 power-sum equation was identified **after** my successful Round 10 boundary query. The portal's legal maximum coordinate was **0.999999**, not 1.000000. This check ensures the comparison uses the actual permitted setting, avoiding a misleading mathematical corner value.

In [ ]:
legal_corner = np.full(4, 0.999999)
f5_model = f5_at(legal_corner)
f5_portal = 8662.405001248297
assert abs(f5_model - f5_portal) < 1e-9
print('Permitted input:', [float(x) for x in legal_corner])
print('Reconstructed value:', f"{f5_model:.12f}")
print('Confirmed portal value:', f"{f5_portal:.12f}")

## 7. Limits and next steps

The plots and verified values establish **what was observed**. They do not prove global optima for unresolved functions or recreate unrecorded pre-submission forecasts. F2/F3/F4/F6 have no verified complete equation, F1's full historical plotting parameters are not retained, and the R11 F7 input remains unavailable.

The historical HEBO-inspired and TuRBO-style code is discussed in [the algorithm guide](../docs/optimisation_algorithms.md). Separate *post-capstone* full implementations are in [benchmarks](../benchmarks/full_optimisers/README.md); any F8 scores generated by those engines are **model simulations**, not portal outcomes.

**To reproduce:** install [the notebook dependencies](../requirements-notebook.txt), then use **Run All**. The CI workflow executes this notebook on a clean machine and publishes the executed notebook as an artifact. Refer to the [datasheet](../docs/final_datasheet.md), [model card](../docs/final_model_card.md) and [assessment checklist](../docs/assessment_25_3.md) for evidence and limitations.